# The University of Azad Jammu and Kashmir
Department of Software Engineering — Digital Image Processing Lab

## Lab Task # 04 – Geometric Transformations with Custom Inputs

| | |
|---|---|
| Name | Sahadat Ali |
| Roll No | 2024-SE-36 |
| Lab Instructor | Engr. Fahad Nisar |
| Semester / Session | Fifth / 2024-28 (SE) |
| Assigned / Due | November 05, 2026 / November 12, 2026, 11:59 PM |

## Objective

Create a **reusable image transformation function** named `transform` that accepts an image, an operation name and that operation's custom parameters, validates every input, and returns the correctly transformed image. The notebook covers five operations (scaling, cropping, flipping, translation, rotation), user-driven prompts, a full validation suite, interpolation comparison, identity checks and short explanations.

**Contents**
1. Libraries and settings
2. Task 1 – Load and validate the image
3. Task 2 – The reusable `transform` function
4. Task 3 – Selecting an operation and entering parameters (prompts)
5. Task 4 – Demonstration of every operation with custom values
6. Task 5 – Interpolation comparison on an enlarged patch
7. Task 6 – Rejected inputs (validation demonstration)
8. Task 7 – Identity and correctness checks
9. Task 8 – Concepts explained
10. Saved results and final conclusion

## 1. Required libraries and settings

In [6]:
import hashlib
import math
import numbers
import tempfile
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- Settings -------------------------------------------------------------
IMAGE_PATH = Path("input_image.png")   # input image (relative path, kept in the repository)
OUTPUT_DIR = Path("output")            # saved transformed images go here
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_OUTPUT_DIM = 4096     # declared maximum: no output may be wider or taller than 4096 px
MAX_OFFSET = 100_000      # declared maximum |tx|, |ty| for translation (pixels)

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", None)

print("NumPy:", np.__version__, "| OpenCV:", cv.__version__, "| Matplotlib:", plt.matplotlib.__version__)

NumPy: 2.4.6 | OpenCV: 5.0.0 | Matplotlib: 3.10.0


## 2. Task 1 – Load the image and validate it

`load_image` checks that **(a)** the file exists, **(b)** it decodes successfully, and **(c)** it has non-zero width and height.

**Colour handling (consistent everywhere):** OpenCV decodes images as **BGR**. The image is converted to **RGB** once, right after loading. All processing, display (`matplotlib`) and the `transform` function work on **RGB, `uint8`** arrays. Colours are converted back to BGR only inside `save_image`, because `cv.imwrite` expects BGR.

In [7]:
def load_image(path):
    """Load an image as an RGB uint8 array, validating existence, decoding and size."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Image file not found: '{path}'")
    data = cv.imread(str(path), cv.IMREAD_COLOR)          # BGR, 3 channels
    if data is None:
        raise ValueError(f"File '{path}' exists but could not be decoded as an image.")
    if data.ndim != 3 or data.shape[0] < 1 or data.shape[1] < 1:
        raise ValueError(f"Image '{path}' has zero width or height: shape {data.shape}.")
    return cv.cvtColor(data, cv.COLOR_BGR2RGB)            # consistent colour order: RGB


def save_image(path, rgb_image):
    """Save an RGB array to disk (converted to BGR for OpenCV)."""
    ok = cv.imwrite(str(path), cv.cvtColor(rgb_image, cv.COLOR_RGB2BGR))
    if not ok:
        raise IOError(f"Could not write '{path}'")


original = load_image(IMAGE_PATH)
original_height, original_width = original.shape[:2]

# Keep the original unchanged: make it read-only and remember a fingerprint to verify at the end.
original.flags.writeable = False
ORIGINAL_HASH = hashlib.sha256(original.tobytes()).hexdigest()

print(f"File           : {IMAGE_PATH}")
print(f"Width          : {original_width} px")
print(f"Height         : {original_height} px")
print(f"Array shape    : {original.shape}   (rows = height, columns = width, channels)")
print(f"Data type      : {original.dtype}")
print(f"Colour order   : RGB")
print(f"SHA-256 (start): {ORIGINAL_HASH[:16]}...")

plt.figure(figsize=(6, 4.2))
plt.imshow(original)
plt.title(f"Original image  ({original_width} x {original_height} px)")
plt.xlabel("x (columns)"); plt.ylabel("y (rows)")
plt.show()

FileNotFoundError: Image file not found: 'input_image.png'

**Validation of the image loader** – a missing file, a corrupt file and an empty array are rejected with clear messages (no unhandled crash):

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    corrupt = Path(tmp) / "corrupt.png"
    corrupt.write_text("this is not image data")          # exists, but cannot be decoded
    for label, path in [("missing file", Path("no_such_image.png")), ("corrupt file", corrupt)]:
        try:
            load_image(path)
            print(f"{label:13s}: UNEXPECTEDLY ACCEPTED")
        except (FileNotFoundError, ValueError) as err:
            print(f"{label:13s}: rejected -> {type(err).__name__}: {err}".replace(tmp, "<tmp>"))

## 3. Task 2 – The reusable `transform` function

`transform(image, operation, **params)` is the **single entry point**. It validates the image, the operation name and the parameters, then returns a **new** array (the input is never modified).

| Operation | Required parameters | Notes |
|---|---|---|
| `"scale"` | `scale_x`, `scale_y`, `interpolation` | factors finite and > 0; output size = `round(width*scale_x)` × `round(height*scale_y)`, each ≥ 1 and ≤ `MAX_OUTPUT_DIM`; interpolation ∈ nearest, bilinear, bicubic, lanczos |
| `"crop"` | `left`, `top`, `right`, `bottom` | integers, `0 ≤ left < right ≤ width`, `0 ≤ top < bottom ≤ height`; **right and bottom are exclusive** |
| `"flip"` | `direction` | `"horizontal"` (mirror left↔right), `"vertical"` (top↔bottom) or `"both"` |
| `"translate"` | `tx`, `ty` | finite numbers (positive, negative or zero), `|tx|,|ty| ≤ MAX_OFFSET`; canvas size is kept; **+tx moves right, +ty moves down**; exposed area is filled with **black** |
| `"rotate"` | `angle`, `expand` | any finite angle in degrees (negative and zero allowed); `expand` must be `True`/`False`; rotation about the image centre; **a positive angle rotates counter-clockwise** (as seen on screen); exposed area is black |

Missing, unknown or extra parameters are rejected — nothing is silently replaced by a default. Translation and rotation use bilinear interpolation internally (this is fixed and documented, it is not one of the user inputs in the task table).

In [ ]:
class TransformError(ValueError):
    """Raised for any invalid image, operation name or parameter."""


OPERATIONS = ("scale", "crop", "flip", "translate", "rotate")
REQUIRED_PARAMS = {
    "scale":     ("scale_x", "scale_y", "interpolation"),
    "crop":      ("left", "top", "right", "bottom"),
    "flip":      ("direction",),
    "translate": ("tx", "ty"),
    "rotate":    ("angle", "expand"),
}
INTERPOLATIONS = {
    "nearest":  cv.INTER_NEAREST,
    "bilinear": cv.INTER_LINEAR,
    "bicubic":  cv.INTER_CUBIC,
    "lanczos":  cv.INTER_LANCZOS4,
}
FLIP_CODES = {"horizontal": 1, "vertical": 0, "both": -1}   # cv.flip codes


# ---------------------------- validation helpers ----------------------------
def _check_image(image):
    if not isinstance(image, np.ndarray):
        raise TransformError(f"image must be a NumPy array, got {type(image).__name__}.")
    if image.dtype != np.uint8:
        raise TransformError(f"image must have dtype uint8, got {image.dtype}.")
    if image.ndim not in (2, 3) or (image.ndim == 3 and image.shape[2] not in (1, 3, 4)):
        raise TransformError(f"image must be 2-D (gray) or H x W x 1/3/4, got shape {image.shape}.")
    if image.shape[0] < 1 or image.shape[1] < 1:
        raise TransformError(f"image is empty (shape {image.shape}); width and height must be at least 1.")


def _finite_number(name, value):
    """Return value as float; reject non-numbers, booleans, NaN and infinity."""
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, numbers.Real):
        raise TransformError(f"'{name}' must be a number, got {value!r} ({type(value).__name__}).")
    value = float(value)
    if not math.isfinite(value):
        raise TransformError(f"'{name}' must be finite, got {value}.")
    return value


def _integer(name, value):
    """Return value as int; reject non-integers (including 10.5 and booleans)."""
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, numbers.Integral):
        raise TransformError(f"'{name}' must be an integer, got {value!r} ({type(value).__name__}).")
    return int(value)


def _choice(name, value, options):
    if not isinstance(value, str) or value.strip().lower() not in options:
        raise TransformError(f"'{name}' must be one of {sorted(options)}, got {value!r}.")
    return value.strip().lower()


def _output_size_ok(width, height):
    if width < 1 or height < 1:
        raise TransformError(f"output size {width} x {height} px would be an empty image (minimum is 1 x 1).")
    if width > MAX_OUTPUT_DIM or height > MAX_OUTPUT_DIM:
        raise TransformError(f"output size {width} x {height} px exceeds the maximum of {MAX_OUTPUT_DIM} x {MAX_OUTPUT_DIM}.")


def _round_half_up(x):
    return int(math.floor(x + 0.5))


# --------------------------- affine matrix helpers --------------------------
def translation_matrix(tx, ty):
    """2x3 affine matrix [[1,0,tx],[0,1,ty]] (x to the right, y downward)."""
    return np.float64([[1, 0, tx], [0, 1, ty]])


def rotation_matrix(width, height, angle, expand):
    """2x3 affine matrix for a rotation about the image centre and the output canvas size.
    Positive angle = counter-clockwise on screen (image origin is the top-left corner)."""
    cx, cy = (width - 1) / 2, (height - 1) / 2          # centre in pixel-index coordinates
    theta = math.fmod(angle, 360.0)                      # same rotation, avoids huge-angle noise
    matrix = cv.getRotationMatrix2D((cx, cy), theta, 1.0)
    if not expand:
        return matrix, (width, height)
    rad = math.radians(theta)
    c, s = abs(math.cos(rad)), abs(math.sin(rad))
    new_w = math.ceil(round(width * c + height * s, 6))  # rounding removes 1e-16 noise (e.g. at 90 degrees)
    new_h = math.ceil(round(width * s + height * c, 6))
    _output_size_ok(new_w, new_h)
    matrix[0, 2] += (new_w - 1) / 2 - cx                 # move the old centre to the new centre
    matrix[1, 2] += (new_h - 1) / 2 - cy
    return matrix, (new_w, new_h)


# ------------------------------ the operations ------------------------------
def _scale(image, scale_x, scale_y, interpolation):
    sx = _finite_number("scale_x", scale_x)
    sy = _finite_number("scale_y", scale_y)
    if sx <= 0 or sy <= 0:
        raise TransformError(f"scale factors must be greater than zero, got scale_x={sx}, scale_y={sy}.")
    method = _choice("interpolation", interpolation, INTERPOLATIONS)
    h, w = image.shape[:2]
    raw_w, raw_h = w * sx, h * sy
    if raw_w > 1e9 or raw_h > 1e9:                        # protects int() and memory before rounding
        raise TransformError(f"output size would be about {raw_w:.3g} x {raw_h:.3g} px, which exceeds the maximum of {MAX_OUTPUT_DIM}.")
    new_w, new_h = _round_half_up(raw_w), _round_half_up(raw_h)
    _output_size_ok(new_w, new_h)
    return cv.resize(image, (new_w, new_h), interpolation=INTERPOLATIONS[method])   # dsize is (width, height)


def _crop(image, left, top, right, bottom):
    left, top = _integer("left", left), _integer("top", top)
    right, bottom = _integer("right", right), _integer("bottom", bottom)
    h, w = image.shape[:2]
    if not (0 <= left < right <= w):
        raise TransformError(f"invalid horizontal range: require 0 <= left < right <= {w}, got left={left}, right={right}.")
    if not (0 <= top < bottom <= h):
        raise TransformError(f"invalid vertical range: require 0 <= top < bottom <= {h}, got top={top}, bottom={bottom}.")
    return image[top:bottom, left:right].copy()          # right/bottom exclusive; copy => independent of the original


def _flip(image, direction):
    direction = _choice("direction", direction, FLIP_CODES)
    return cv.flip(image, FLIP_CODES[direction])


def _translate(image, tx, ty):
    tx, ty = _finite_number("tx", tx), _finite_number("ty", ty)
    if abs(tx) > MAX_OFFSET or abs(ty) > MAX_OFFSET:
        raise TransformError(f"|tx| and |ty| must be at most {MAX_OFFSET} px, got tx={tx}, ty={ty}.")
    h, w = image.shape[:2]
    return cv.warpAffine(image, translation_matrix(tx, ty), (w, h), flags=cv.INTER_LINEAR,
                         borderMode=cv.BORDER_CONSTANT, borderValue=0)       # canvas size kept, black background


def _rotate(image, angle, expand):
    angle = _finite_number("angle", angle)
    if not isinstance(expand, (bool, np.bool_)):
        raise TransformError(f"'expand' must be True or False, got {expand!r} ({type(expand).__name__}).")
    h, w = image.shape[:2]
    matrix, (new_w, new_h) = rotation_matrix(w, h, angle, bool(expand))
    return cv.warpAffine(image, matrix, (new_w, new_h), flags=cv.INTER_LINEAR,
                         borderMode=cv.BORDER_CONSTANT, borderValue=0)


_DISPATCH = {"scale": _scale, "crop": _crop, "flip": _flip, "translate": _translate, "rotate": _rotate}


# ------------------------------- entry point --------------------------------
def transform(image, operation, **params):
    """Apply one geometric operation with custom parameters and return the new image.

    operation : "scale" | "crop" | "flip" | "translate" | "rotate"
    params    : see REQUIRED_PARAMS.  Raises TransformError for any invalid input.
    """
    _check_image(image)
    if not isinstance(operation, str) or operation.strip().lower() not in OPERATIONS:
        raise TransformError(f"unknown operation {operation!r}; supported operations: {list(OPERATIONS)}.")
    op = operation.strip().lower()
    missing = [p for p in REQUIRED_PARAMS[op] if params.get(p) is None]
    if missing:
        raise TransformError(f"operation '{op}' is missing required parameter(s): {missing}.")
    extra = [p for p in params if p not in REQUIRED_PARAMS[op]]
    if extra:
        raise TransformError(f"operation '{op}' does not accept parameter(s) {extra}; expected {list(REQUIRED_PARAMS[op])}.")
    return _DISPATCH[op](image, **params)


print("transform() defined. Supported operations:", list(OPERATIONS))
print(f"Declared limits: output <= {MAX_OUTPUT_DIM} x {MAX_OUTPUT_DIM} px, |tx|,|ty| <= {MAX_OFFSET} px")

## 4. Task 3 – Selecting an operation and entering parameters

The prompt layer asks for the operation and for each of its parameters, **checks the typed text** (non-numeric text, NaN/infinity, non-integers, unsupported choices) and passes the values to `transform`. If `transform` rejects the values (for example an invalid crop region), the error is shown and the values are asked again — the notebook never crashes on bad input. Nothing is hard-coded inside `transform`.

`INTERACTIVE = True` makes the prompts use the notebook's real `input()` box. So that the notebook also runs top-to-bottom with *Run All* and the results are reproducible, the demo below feeds **pre-filled answers** through exactly the same prompts (they are printed with the tag `<- scripted`). A live session is available in the last cell of this section.

In [ ]:
INTERACTIVE = False        # False: answers come from SCRIPTED_ANSWERS; True: real input() prompts
SCRIPTED_ANSWERS = []      # queue of pre-filled answers used when INTERACTIVE is False


def read_value(prompt):
    """Read one answer: from the notebook input box, or from the scripted queue."""
    if INTERACTIVE:
        return input(prompt)
    if not SCRIPTED_ANSWERS:
        raise RuntimeError("scripted answers exhausted (the demo script is shorter than the prompts asked).")
    answer = SCRIPTED_ANSWERS.pop(0)
    print(f"{prompt}{answer}   <- scripted")
    return answer


# ---- text parsers: turn typed text into values, raising TransformError for bad text ----
def parse_number(text, name):
    try:
        value = float(text.strip())
    except ValueError:
        raise TransformError(f"'{text}' is not a number for {name}.") from None
    if not math.isfinite(value):
        raise TransformError(f"{name} must be finite, got '{text}'.")
    return value


def parse_integer(text, name):
    try:
        return int(text.strip())
    except ValueError:
        raise TransformError(f"'{text}' is not a whole number for {name} (e.g. 120).") from None


def parse_choice(options):
    def parser(text, name):
        value = text.strip().lower()
        if value not in options:
            raise TransformError(f"'{text}' is not a valid {name}; choose one of {sorted(options)}.")
        return value
    return parser


def parse_yes_no(text, name):
    value = text.strip().lower()
    if value in ("y", "yes", "true", "t", "1"):
        return True
    if value in ("n", "no", "false", "f", "0"):
        return False
    raise TransformError(f"'{text}' is not valid for {name}; answer yes or no.")


PARAMETER_PROMPTS = {
    "scale":     [("scale_x", parse_number, "Horizontal scale factor (> 0, e.g. 1.5)"),
                  ("scale_y", parse_number, "Vertical scale factor (> 0, e.g. 0.5)"),
                  ("interpolation", parse_choice(INTERPOLATIONS), "Interpolation (nearest/bilinear/bicubic/lanczos)")],
    "crop":      [("left", parse_integer, "Left (integer)"), ("top", parse_integer, "Top (integer)"),
                  ("right", parse_integer, "Right, exclusive (integer)"), ("bottom", parse_integer, "Bottom, exclusive (integer)")],
    "flip":      [("direction", parse_choice(FLIP_CODES), "Direction (horizontal/vertical/both)")],
    "translate": [("tx", parse_number, "Horizontal offset tx in pixels (+ right, - left)"),
                  ("ty", parse_number, "Vertical offset ty in pixels (+ down, - up)")],
    "rotate":    [("angle", parse_number, "Angle in degrees (+ counter-clockwise, - clockwise)"),
                  ("expand", parse_yes_no, "Expand the canvas to fit the whole rotated image? (yes/no)")],
}


def run_session(image):
    """Ask for an operation and its parameters, call transform, retry on invalid input.
    Returns (operation, parameters, result), or None if the user types q at the operation prompt."""
    while True:
        op = read_value(f"Operation {list(OPERATIONS)} (q = quit): ").strip().lower()
        if op == "q":
            return None
        if op in OPERATIONS:
            break
        print(f"  [x] unknown operation '{op}'. Please choose one of {list(OPERATIONS)}.")
    while True:
        params = {}
        for name, parser, hint in PARAMETER_PROMPTS[op]:
            while True:                                   # re-ask this one field until the text itself is valid
                raw = read_value(f"  {hint}: ")
                try:
                    params[name] = parser(raw, name)
                    break
                except TransformError as err:
                    print(f"  [x] {err}  Please try again.")
        try:
            return op, params, transform(image, op, **params)
        except TransformError as err:                     # e.g. crop region outside the image
            print(f"  [x] transform rejected the values: {err}")
            print("      Please re-enter the parameters.")


def describe(params):
    return ", ".join(f"{k}={v}" for k, v in params.items())


print("Prompt layer ready (INTERACTIVE =", INTERACTIVE, ")")

**Prompt demo.** Each session below uses the prompts above with pre-filled answers; the first sessions deliberately contain a **wrong answer followed by a correction** (non-numeric text, an unsupported choice, an invalid crop region) to show that bad input is reported and can be fixed.

In [ ]:
def figure_pair(original_img, result_img, title, filename):
    """Original (left) and result (right), labelled with operation, parameters and output size; also saves the result."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].imshow(original_img)
    axes[0].set_title(f"Original\n{original_img.shape[1]} x {original_img.shape[0]} px", fontsize=10)
    axes[1].imshow(result_img)
    axes[1].set_title(f"{title}\noutput {result_img.shape[1]} x {result_img.shape[0]} px", fontsize=10)
    for ax in axes:
        ax.set_xlabel("x"); ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()
    if filename:
        save_image(OUTPUT_DIR / filename, result_img)


SAVED = {}      # filename -> shape, filled as results are saved
def show_result(op, params, result, filename):
    title = f"{op}  ({describe(params)})"
    figure_pair(original, result, title, filename)
    SAVED[filename] = result.shape
    print(f"saved -> {OUTPUT_DIR / filename}   | operation={op} | parameters: {describe(params)} | output {result.shape[1]} x {result.shape[0]} px")


demo_sessions = [
    # (scripted answers, file to save)
    (["scale", "1.5", "abc", "0.8", "bogus", "bicubic"], "prompt_scale.png"),                        # bad number + bad choice, corrected
    (["crop", "300", "50", "100", "250", "60", "40", "390", "260"], "prompt_crop.png"),               # left>right rejected by transform, corrected
    (["shear", "rotate", "-25", "maybe", "yes"], "prompt_rotate.png"),                                # unknown operation + bad yes/no, corrected
]
for answers, fname in demo_sessions:
    print("=" * 78)
    SCRIPTED_ANSWERS[:] = answers
    op, params, result = run_session(original)
    show_result(op, params, result, fname)
assert not SCRIPTED_ANSWERS, "unused scripted answers left over"

**Live session (optional).** Set `RUN_LIVE_SESSION = True` and run this cell to select an operation and type your own values in the notebook input box. It is `False` by default so that *Run All* does not wait for typing.

In [ ]:
RUN_LIVE_SESSION = False

if RUN_LIVE_SESSION:
    INTERACTIVE = True
    try:
        outcome = run_session(original)
        if outcome is not None:
            live_op, live_params, live_result = outcome
            figure_pair(original, live_result, f"{live_op}  ({describe(live_params)})", None)
    finally:
        INTERACTIVE = False
else:
    print("Live session skipped (RUN_LIVE_SESSION = False). Set it to True to enter your own values.")

## 5. Task 4 – Demonstration of every operation with custom values

Each result is shown next to the original with the **operation, parameters and output dimensions** in the title, and is **saved** to the `output/` folder. All calls go through the same `transform` function.

### 5.1 Scaling – enlargement, reduction, proportional and non-proportional

In [ ]:
scale_cases = [
    ("enlarge, proportional",      dict(scale_x=1.5, scale_y=1.5, interpolation="bicubic"),  "scale_enlarge_proportional.png"),
    ("reduce, proportional",       dict(scale_x=0.5, scale_y=0.5, interpolation="bilinear"), "scale_reduce_proportional.png"),
    ("non-proportional (distorts)", dict(scale_x=1.5, scale_y=0.6, interpolation="lanczos"),  "scale_nonproportional.png"),
]
rows = []
for label, params, fname in scale_cases:
    result = transform(original, "scale", **params)
    print(f"--- {label} ---")
    show_result("scale", params, result, fname)
    rows.append({"case": label, "scale_x": params["scale_x"], "scale_y": params["scale_y"],
                 "output (W x H)": f"{result.shape[1]} x {result.shape[0]}",
                 "aspect ratio W/H": round(result.shape[1] / result.shape[0], 3)})
rows.insert(0, {"case": "original", "scale_x": 1, "scale_y": 1, "output (W x H)": f"{original_width} x {original_height}",
                "aspect ratio W/H": round(original_width / original_height, 3)})
pd.DataFrame(rows)

**Observation.** Equal factors (1.5 and 0.5) keep the aspect ratio W/H at about 1.50 (1.504 and 1.507 versus 1.503 originally; the tiny differences come only from rounding the output size to whole pixels), so the cat is only larger or smaller. With different factors (1.5 horizontally, 0.6 vertically) the ratio changes to 3.761 and the cat looks stretched sideways — this is **aspect-ratio distortion**.

### 5.2 Cropping – exclusive right/bottom coordinates

In [ ]:
params = dict(left=100, top=50, right=350, bottom=250)
cropped = transform(original, "crop", **params)
show_result("crop", params, cropped, "crop_region.png")
print("Expected size:", (params["right"] - params["left"]), "x", (params["bottom"] - params["top"]),
      "-> right and bottom are exclusive, so the last pixel kept is column", params["right"] - 1, "and row", params["bottom"] - 1)

### 5.3 Flipping – horizontal, vertical and both

In [ ]:
flips = {d: transform(original, "flip", direction=d) for d in ("horizontal", "vertical", "both")}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(original); axes[0].set_title(f"Original\n{original_width} x {original_height} px", fontsize=10)
for ax, (d, img) in zip(axes[1:], flips.items()):
    ax.imshow(img)
    ax.set_title(f"flip (direction={d})\noutput {img.shape[1]} x {img.shape[0]} px", fontsize=10)
plt.tight_layout()
plt.show()
for d, img in flips.items():
    save_image(OUTPUT_DIR / f"flip_{d}.png", img)
    SAVED[f"flip_{d}.png"] = img.shape
print("saved ->", ", ".join(f"flip_{d}.png" for d in flips))

### 5.4 Translation – shifts in opposite directions

Positive `tx` moves the image **right**, positive `ty` moves it **down** (the y-axis points downward). The affine matrix used is `[[1, 0, tx], [0, 1, ty]]`.

In [ ]:
for params, fname in [(dict(tx=80, ty=40), "translate_plus.png"), (dict(tx=-80, ty=-40), "translate_minus.png")]:
    result = transform(original, "translate", **params)
    print("Affine matrix:\n", translation_matrix(**params))
    show_result("translate", params, result, fname)
    tx, ty = params["tx"], params["ty"]
    print(f"Clipped: {abs(tx)} column(s) on the {'right' if tx > 0 else 'left'} edge and {abs(ty)} row(s) on the {'bottom' if ty > 0 else 'top'} edge were pushed outside the canvas and lost.")
    print(f"Exposed: the same-sized strips on the {'left' if tx > 0 else 'right'} and {'top' if ty > 0 else 'bottom'} edges are filled with black background.\n")

### 5.5 Rotation – positive and negative angles, with and without canvas expansion

A **positive angle rotates counter-clockwise** (as seen on screen), a negative angle clockwise, about the image centre. With `expand=False` the canvas keeps its size and the corners are clipped; with `expand=True` the canvas grows to contain the whole rotated image.

In [ ]:
rotation_cases = [
    (dict(angle=30,  expand=False), "rotate_plus30_noexpand.png"),
    (dict(angle=30,  expand=True),  "rotate_plus30_expand.png"),
    (dict(angle=-45, expand=False), "rotate_minus45_noexpand.png"),
    (dict(angle=-45, expand=True),  "rotate_minus45_expand.png"),
]
for params, fname in rotation_cases:
    result = transform(original, "rotate", **params)
    show_result("rotate", params, result, fname)
M, size = rotation_matrix(original_width, original_height, 30, True)
print("Affine matrix for angle=+30, expand=True (output canvas", size, "):\n", np.round(M, 4))

## 6. Task 5 – Interpolation comparison on the same enlarged patch

A small 24 x 18 pixel patch (pupil highlight and eyelid edge) is cut out with `transform(..., "crop")` and enlarged **16 times** with each of the four interpolation methods using `transform(..., "scale")`. All four results come from the same patch and have the same output size.

In [ ]:
patch_box = dict(left=164, top=98, right=188, bottom=116)
patch = transform(original, "crop", **patch_box)
zoomed = {m: transform(patch, "scale", scale_x=16, scale_y=16, interpolation=m)
          for m in ("nearest", "bilinear", "bicubic", "lanczos")}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].imshow(original); axes[0].set_title("Where the patch comes from", fontsize=10)
axes[0].add_patch(plt.Rectangle((patch_box["left"] - 0.5, patch_box["top"] - 0.5),
                                patch_box["right"] - patch_box["left"], patch_box["bottom"] - patch_box["top"],
                                fill=False, edgecolor="yellow", linewidth=2))
axes[1].imshow(patch); axes[1].set_title(f"Patch ({patch.shape[1]} x {patch.shape[0]} px)", fontsize=10)
axes[2].imshow(zoomed["nearest"]); axes[2].set_title(f"Enlarged x16 -> {zoomed['nearest'].shape[1]} x {zoomed['nearest'].shape[0]} px", fontsize=10)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 7.6))
for ax, (m, img) in zip(axes.ravel(), zoomed.items()):
    ax.imshow(img)
    ax.set_title(f"{m}  (scale 16 x 16, output {img.shape[1]} x {img.shape[0]} px)", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

for m, img in zoomed.items():
    save_image(OUTPUT_DIR / f"interp_{m}.png", img)
    SAVED[f"interp_{m}.png"] = img.shape

def edge_strength(img):        # 99th percentile of the Sobel gradient magnitude of the gray image (larger = crisper edges)
    gray = cv.cvtColor(img, cv.COLOR_RGB2GRAY).astype(np.float64)
    return float(np.percentile(np.hypot(cv.Sobel(gray, cv.CV_64F, 1, 0), cv.Sobel(gray, cv.CV_64F, 0, 1)), 99))

lo, hi = patch.min(axis=(0, 1)), patch.max(axis=(0, 1))          # per-channel value range of the original patch
def overshoot(img):            # share of pixels outside the original patch's value range (numerical sign of ringing)
    return float(np.mean(np.any((img < lo) | (img > hi), axis=2)))

pd.DataFrame([{"method": m, "edge strength (p99 gradient)": round(edge_strength(i), 1),
               "identical neighbour pixels": f"{100 * float(np.mean(np.all(i[:, 1:] == i[:, :-1], axis=2))):.1f} %",
               "pixels outside original value range": f"{100 * overshoot(i):.2f} %"} for m, i in zoomed.items()])

**Observations (from the enlarged patches and the table above).**
- **Nearest-neighbour** copies the closest source pixel, so every source pixel becomes a 16 x 16 block: edges are hard, **blocky and jagged**, the table shows by far the most identical neighbouring pixels, and it never creates new colours.
- **Bilinear** blends the 4 nearest pixels: the blocks disappear and the result is smooth, but it has the lowest edge strength of the smooth methods, so the highlight and the eyelid edge look **softer/blurrier**.
- **Bicubic** (4 x 4 neighbourhood) and **Lanczos** (8 x 8 neighbourhood) give visibly **crisper** edges than bilinear; their edge strength is higher and the two look very similar, Lanczos being slightly crisper in the table.
- Bicubic and Lanczos also produce a small share of pixels slightly **outside the original value range** (the numerical sign of mild **ringing/overshoot** next to strong edges), which bilinear and nearest-neighbour can never produce.

## 7. Task 6 – Rejected inputs

`expect_rejection` calls `transform` with an invalid input and records the error message. If an invalid input were ever accepted, the table would show **ACCEPTED (FAIL)**. At least one rejected input is shown for **every** operation, plus unknown operations, missing/extra parameters and empty/invalid images.

In [ ]:
rejection_log = []

def expect_rejection(operation_group, description, image, operation, **params):
    try:
        transform(image, operation, **params)
        outcome, message = "ACCEPTED (FAIL)", "-"
    except TransformError as err:
        outcome, message = "Rejected", str(err)
    rejection_log.append({"operation": operation_group, "invalid input": description, "result": outcome, "error message": message})

img = original
# --- scaling
expect_rejection("scale", "scale_x = 0",                         img, "scale", scale_x=0, scale_y=1, interpolation="bilinear")
expect_rejection("scale", "scale_y = -2 (negative)",             img, "scale", scale_x=1, scale_y=-2, interpolation="bilinear")
expect_rejection("scale", "scale_x = NaN",                       img, "scale", scale_x=float("nan"), scale_y=1, interpolation="bilinear")
expect_rejection("scale", "scale_y = 'abc' (non-numeric)",       img, "scale", scale_x=1, scale_y="abc", interpolation="bilinear")
expect_rejection("scale", "interpolation = 'area' (unsupported)", img, "scale", scale_x=2, scale_y=2, interpolation="area")
expect_rejection("scale", "scale 0.001 -> 0 x 0 px (empty image)", img, "scale", scale_x=0.001, scale_y=0.001, interpolation="nearest")
expect_rejection("scale", "scale 20 -> 9020 px wide (over maximum)", img, "scale", scale_x=20, scale_y=20, interpolation="nearest")
expect_rejection("scale", "interpolation missing",               img, "scale", scale_x=2, scale_y=2)
# --- cropping
expect_rejection("crop", "left=300 >= right=100",                img, "crop", left=300, top=10, right=100, bottom=100)
expect_rejection("crop", "right=500 > width 451",                img, "crop", left=0, top=0, right=500, bottom=100)
expect_rejection("crop", "left = -5 (negative)",                 img, "crop", left=-5, top=0, right=100, bottom=100)
expect_rejection("crop", "bottom = 300 but top = 300 (empty)",   img, "crop", left=0, top=300, right=100, bottom=300)
expect_rejection("crop", "left = 10.5 (not an integer)",         img, "crop", left=10.5, top=0, right=100, bottom=100)
# --- flipping
expect_rejection("flip", "direction = 'diagonal' (unsupported)", img, "flip", direction="diagonal")
expect_rejection("flip", "direction = 5 (not a string)",         img, "flip", direction=5)
# --- translation
expect_rejection("translate", "tx = NaN",                        img, "translate", tx=float("nan"), ty=0)
expect_rejection("translate", "ty = infinity",                   img, "translate", tx=0, ty=float("inf"))
expect_rejection("translate", "tx = 'abc' (non-numeric)",        img, "translate", tx="abc", ty=0)
expect_rejection("translate", "tx = 1e9 (over maximum offset)",  img, "translate", tx=1e9, ty=0)
# --- rotation
expect_rejection("rotate", "angle = 'abc' (non-numeric)",        img, "rotate", angle="abc", expand=True)
expect_rejection("rotate", "angle = infinity",                   img, "rotate", angle=float("inf"), expand=True)
expect_rejection("rotate", "angle = NaN",                        img, "rotate", angle=float("nan"), expand=False)
expect_rejection("rotate", "expand = 'maybe' (not True/False)",  img, "rotate", angle=10, expand="maybe")
# --- general
expect_rejection("general", "unknown operation 'shear'",         img, "shear", factor=0.3)
expect_rejection("general", "missing parameter (crop without bottom)", img, "crop", left=0, top=0, right=10)
expect_rejection("general", "unexpected parameter (flip with angle)",  img, "flip", direction="both", angle=10)
expect_rejection("general", "operation name is not a string",    img, 42)
expect_rejection("general", "empty image (0 x 5 x 3 array)",     np.zeros((0, 5, 3), np.uint8), "flip", direction="both")
expect_rejection("general", "image is None",                     None, "flip", direction="both")

rejections = pd.DataFrame(rejection_log)
assert (rejections["result"] == "Rejected").all(), "an invalid input was accepted!"
print(f"{len(rejections)} invalid inputs tried -> {int((rejections['result'] == 'Rejected').sum())} rejected with a clear message, 0 accepted.")
rejections

**Valid boundary inputs are accepted** (zero and negative offsets/angles, whole-image crop, 1-pixel results, exactly the maximum size) — they must *not* be rejected:

In [ ]:
boundary_cases = [
    ("translate", dict(tx=0, ty=0)),
    ("translate", dict(tx=-25, ty=0)),
    ("translate", dict(tx=0.5, ty=-0.5)),
    ("rotate",    dict(angle=0, expand=False)),
    ("rotate",    dict(angle=-90, expand=True)),
    ("rotate",    dict(angle=720, expand=True)),
    ("crop",      dict(left=0, top=0, right=original_width, bottom=original_height)),
    ("crop",      dict(left=0, top=0, right=1, bottom=1)),
    ("scale",     dict(scale_x=1 / original_width, scale_y=1 / original_height, interpolation="bilinear")),
    ("scale",     dict(scale_x=MAX_OUTPUT_DIM / original_width, scale_y=1, interpolation="nearest")),
    ("flip",      dict(direction="HORIZONTAL")),
]
rows = []
for op, params in boundary_cases:
    out = transform(original, op, **params)
    rows.append({"operation": op, "parameters": describe(params), "accepted": "yes", "output (W x H)": f"{out.shape[1]} x {out.shape[0]}"})
pd.DataFrame(rows)

## 8. Task 7 – Identity checks and correctness checks

### 8.1 Identity cases (required)
Zero translation, scale factors of one and zero rotation must return an image **identical** to the original. *Actual behaviour* is measured: the output shape and the maximum absolute pixel difference from the original.

In [ ]:
identity_rows = []
def identity_check(name, expected, op, **params):
    out = transform(original, op, **params)
    same = out.shape == original.shape and np.array_equal(out, original)
    diff = int(np.abs(out.astype(int) - original.astype(int)).max()) if out.shape == original.shape else "n/a"
    identity_rows.append({"test": name, "expected behaviour": expected,
                          "actual behaviour": f"shape {out.shape}, max |difference| = {diff}",
                          "result": "Pass" if same else "Fail"})

identity_check("translate tx=0, ty=0", "output identical to original", "translate", tx=0, ty=0)
for m in INTERPOLATIONS:
    identity_check(f"scale 1 x 1 ({m})", "same size, identical pixels", "scale", scale_x=1, scale_y=1, interpolation=m)
identity_check("rotate 0 deg, expand=False", "output identical to original", "rotate", angle=0, expand=False)
identity_check("rotate 0 deg, expand=True",  "same canvas, identical pixels", "rotate", angle=0, expand=True)
identity_check("rotate 360 deg, expand=True", "full turn = identity", "rotate", angle=360, expand=True)
identity_table = pd.DataFrame(identity_rows)
print("Identity checks passed:", int((identity_table["result"] == "Pass").sum()), "of", len(identity_table))
identity_table

### 8.2 Additional correctness checks
These compare `transform` with **independent** NumPy references, to confirm the operations are not just running but producing the right pixels (including the rotation direction and the exclusive crop boundary).

In [ ]:
H, W = original.shape[:2]
checks = []
def check(name, expected, condition):
    checks.append({"check": name, "expected": expected, "result": "Pass" if bool(condition) else "Fail"})

# crop: pixel mapping and exclusive boundary
c = transform(original, "crop", left=100, top=50, right=350, bottom=250)
check("crop shape", "(200, 250, 3) = (bottom-top, right-left)", c.shape == (200, 250, 3))
check("crop first pixel", "cropped[0,0] == original[top,left]", np.array_equal(c[0, 0], original[50, 100]))
check("crop last pixel (exclusive)", "cropped[-1,-1] == original[bottom-1,right-1]", np.array_equal(c[-1, -1], original[249, 349]))
# flips vs NumPy
check("flip horizontal", "== original[:, ::-1]", np.array_equal(transform(original, "flip", direction="horizontal"), original[:, ::-1]))
check("flip vertical",   "== original[::-1, :]", np.array_equal(transform(original, "flip", direction="vertical"), original[::-1, :]))
check("flip both",       "== original[::-1, ::-1]", np.array_equal(transform(original, "flip", direction="both"), original[::-1, ::-1]))
# translation vs slicing
t = transform(original, "translate", tx=80, ty=40)
check("translate (+80,+40) content", "t[40:, 80:] == original[:-40, :-80]", np.array_equal(t[40:, 80:], original[:-40, :-80]))
check("translate (+80,+40) background", "exposed top 40 rows and left 80 columns are black", (t[:40] == 0).all() and (t[:, :80] == 0).all())
t = transform(original, "translate", tx=-80, ty=-40)
check("translate (-80,-40) content", "t[:-40, :-80] == original[40:, 80:]", np.array_equal(t[:-40, :-80], original[40:, 80:]))
check("translate canvas size", "output size == input size", t.shape == original.shape)
# single pixel lands where expected (image coordinates x,y  <->  array index [y, x])
x0, y0, tx, ty = 120, 60, 80, 40
t = transform(original, "translate", tx=tx, ty=ty)
check("pixel (x=120,y=60) moves to (x=200,y=100)", "t[100,200] == original[60,120]", np.array_equal(t[y0 + ty, x0 + tx], original[y0, x0]))
# rotation direction and canvas expansion
r90 = transform(original, "rotate", angle=90, expand=True)
check("rotate +90 expand: size", f"canvas becomes ({W}, {H}) rows x cols", r90.shape[:2] == (W, H))
check("rotate +90 == counter-clockwise", "== np.rot90(original, 1)", np.array_equal(r90, np.rot90(original, 1)))
check("rotate -90 == clockwise", "== np.rot90(original, -1)", np.array_equal(transform(original, "rotate", angle=-90, expand=True), np.rot90(original, -1)))
check("rotate 180", "== both flips", np.array_equal(transform(original, "rotate", angle=180, expand=True), original[::-1, ::-1]))
r45 = transform(original, "rotate", angle=45, expand=True)
exp_w = math.ceil(round(W * math.cos(math.radians(45)) + H * math.sin(math.radians(45)), 6))
exp_h = math.ceil(round(W * math.sin(math.radians(45)) + H * math.cos(math.radians(45)), 6))
check("rotate 45 expand: size", f"width = ceil(W|cos|+H|sin|) = {exp_w}, height = {exp_h}", r45.shape[:2] == (exp_h, exp_w))
check("rotate without expand keeps canvas", "output size == input size", transform(original, "rotate", angle=30, expand=False).shape == original.shape)
# scaling sizes
check("scale 1.5 x 1.5", f"{round(W*1.5+1e-9)} x {round(H*1.5)} (W x H)", transform(original, "scale", scale_x=1.5, scale_y=1.5, interpolation="bilinear").shape[:2] == (450, 677))
check("scale non-proportional 1.5 x 0.6", "677 x 180 (W x H)", transform(original, "scale", scale_x=1.5, scale_y=0.6, interpolation="nearest").shape[:2] == (180, 677))
# original untouched by all of the above
check("original unchanged", "SHA-256 equals the fingerprint taken at load time", hashlib.sha256(original.tobytes()).hexdigest() == ORIGINAL_HASH)

correctness_table = pd.DataFrame(checks)
print("Correctness checks passed:", int((correctness_table["result"] == "Pass").sum()), "of", len(correctness_table))
correctness_table

## 9. Task 8 – Concepts explained

### 9.1 Image coordinates versus array indexing
- **Image (Cartesian-style) coordinates** are written **(x, y)**: *x* is the horizontal position (column) and *y* the vertical position (row). The origin (0, 0) is the **top-left** corner, *x* increases to the right and *y* increases **downwards**.
- **NumPy array indexing** is `image[row, column]` = `image[y, x]`, so the order is **swapped**. The array shape is `(height, width, channels)`, i.e. `(rows, columns, channels)`, whereas OpenCV functions such as `cv.resize` and `cv.warpAffine` take sizes as `(width, height)`.
- Cropping therefore uses `image[top:bottom, left:right]`, and the end indices (`bottom`, `right`) are **exclusive**.

In [ ]:
x, y = 120, 60
print(f"Image shape (rows, columns, channels) = {original.shape} -> width = {original.shape[1]}, height = {original.shape[0]}")
print(f"Pixel at image coordinates (x={x}, y={y}) is read as original[{y}, {x}] =", original[y, x])
print(f"Valid x range: 0 .. {original_width - 1}   |   valid y range: 0 .. {original_height - 1}")

### 9.2 Aspect-ratio distortion
The aspect ratio is width / height. Scaling with **equal** factors keeps it (apart from rounding to whole pixels). With **different** `scale_x` and `scale_y` the ratio changes, so shapes are stretched or squeezed — in section 5.1 the ratio changed from 1.503 to 3.761 and the cat appears stretched horizontally. To avoid distortion, use `scale_x == scale_y`.

### 9.3 Rotation canvas expansion
Rotating about the centre keeps the canvas size when `expand=False`, so the **corners are cut off** and empty corners are filled with black. With `expand=True` the output canvas is enlarged to  
`new_width = ceil(width·|cos θ| + height·|sin θ|)`, `new_height = ceil(width·|sin θ| + height·|cos θ|)`,  
so nothing is cut off; the extra area is black background. For example 90° swaps width and height (451 x 300 becomes 300 x 451). A **positive angle rotates counter-clockwise**, a negative angle clockwise (verified in section 8.2).

### 9.4 Translation clipping
Translation keeps the original canvas size. Pixels pushed beyond an edge are **clipped (lost)**, and the strip that the shifted image no longer covers on the opposite side is **exposed background** (black). Shifting by `(tx, ty)` loses `|tx|` columns and `|ty|` rows and exposes strips of the same widths; shifting back later cannot restore the lost pixels.

## 10. Saved results, integrity check and conclusion

In [ ]:
# Reload every saved file to prove it was written correctly, and confirm the original is untouched.
rows = []
for fname, shape in sorted(SAVED.items()):
    path = OUTPUT_DIR / fname
    reloaded = cv.cvtColor(cv.imread(str(path)), cv.COLOR_BGR2RGB)
    rows.append({"file": str(path), "size on disk (KB)": round(path.stat().st_size / 1024, 1),
                 "shape": reloaded.shape, "matches expected": reloaded.shape == shape})
saved_table = pd.DataFrame(rows)
assert saved_table["matches expected"].all()
print(f"{len(saved_table)} transformed images saved in '{OUTPUT_DIR}/' and verified by reloading.")
print("Original unchanged:", hashlib.sha256(original.tobytes()).hexdigest() == ORIGINAL_HASH,
      "| read-only:", not original.flags.writeable)
saved_table

### Conclusion
A single reusable function, `transform(image, operation, **params)`, now performs scaling, cropping, flipping, translation and rotation with **custom values** supplied through notebook prompts. Every input is validated twice (typed text in the prompts, and all parameters again inside `transform`); invalid input produces a clear message and can be corrected, while valid boundary values such as zero or negative offsets and angles are accepted. The checks confirmed that identity cases return the unchanged image, that flips, crops, translations and 90° rotations match independent NumPy results, that a positive angle rotates counter-clockwise, and that the original image is never modified. The interpolation comparison showed nearest-neighbour as blocky, bilinear as smooth but soft, bicubic as smooth and sharper, and Lanczos as the sharpest with possible ringing. All transformed images are saved in the `output/` folder.